# ROMS-TOOLS setup for Iceland3_MARBL_2024

First step is to set up the outer grid using ROMS-TOOLS and save the grid file.

In [5]:
from roms_tools import Grid
import xarray as xr

In [6]:
project='/anvil/projects/x-ees250129/x-uheede/INPUT_files/Iceland3_MARBL_2024/'
datasets='/anvil/projects/x-ees250129/Datasets/'
model_name='Iceland3'
child_name='Iceland4'
grid_path='/anvil/projects/x-ees250129/x-uheede/MATLAB/setup_r2r_phys+bgc/1.Make_grid/Iceland3_grid_MAT.nc'

In [7]:
import dask

dask.config.set(
    scheduler="threads",  # Use multi-threading
    n_workers=10,  # Number of threads; adjust as needed
)

In [ ]:
%%time

grid = Grid(
    nx=640,
    ny=384,
    size_x=32,
    size_y=19.2,
    center_lon=-21.68,
    center_lat=64.325,
    rot=0,
    mask_shapefile=datasets+"GSHHS/gshhg-shp-2.3.7/GSHHS_shp/f/GSHHS_f_L1.shp",
    topography_source={
        "name": "EMOD",
        "path": datasets+"EMODnet_C2.nc"},
    close_narrow_channels=True,
    N=60  # number of vertical layers
)

In [ ]:
#grid = Grid.from_file(grid_path)
grid.plot()

In [ ]:
grid.plot()

In [ ]:
filepath = project+model_name+'_grid.nc'

In [ ]:
#grid.update_vertical_coordinate(N=60, theta_s=5.0, theta_b=5.0, hc=5.0, verbose=False)

In [ ]:
#grid.plot_vertical_coordinate(eta=120, max_nr_layer_contours=20)

In [ ]:
grid.save(filepath)

In [ ]:
grid.ds

In [ ]:
tpxo_path = datasets+"TPXO/TPXO10.v2/"
tpxo_dict = {
    "grid": tpxo_path + "grid_tpxo10v2.nc",
    "h": tpxo_path + "h_tpxo10.v2.nc",
    "u": tpxo_path + "u_tpxo10.v2.nc",
}

Next, we set up tidal forcing:

In [ ]:
from roms_tools import TidalForcing

In [ ]:
from datetime import datetime

In [ ]:
model_reference_date = datetime(2000, 1, 1)

In [ ]:

tidal_forcing = TidalForcing(
    grid=grid,
    source={"name": "TPXO", "path": tpxo_dict},
    ntides=15,  # Number of constituents to consider <= 15. Default is 10.
    model_reference_date=model_reference_date,  # Model reference date. Default is January 1, 2000.
    use_dask=True
)

In [ ]:
filepath = project+model_name+"_tides.nc"

In [ ]:
%time tidal_forcing.save(filepath)

For the surface forcing, we use ERA5 plus the unified BGC dataset

In [ ]:
from roms_tools import Grid, SurfaceForcing

In [ ]:
start_time = datetime(2023, 12, 1)
end_time = datetime(2024, 12, 31)

In [ ]:
surface_forcing_kwargs = {
    "grid": grid,
    "start_time": start_time,
    "end_time": end_time,
    "type": "physics",
    "model_reference_date": datetime(2000, 1, 1), # this is the default
}

In [ ]:
surface_forcing.plot("uwnd", time=0)

In [ ]:
%%time

surface_forcing = SurfaceForcing(
    **surface_forcing_kwargs,
    source={"name": "ERA5"},
    use_dask=True,
)

In [ ]:
#cesm_bgc_path = "/global/cfs/projectdirs/m4746/Datasets/CESM_REGRIDDED/CESM-surface_lowres_regridded.nc"
unified_bgc_path = datasets+"UNIFIED/BGCdataset.nc"

In [ ]:
%%time

unified_bgc_surface_forcing = SurfaceForcing(
    grid=grid,
    start_time=start_time,
    end_time=end_time,
    source={"name": "UNIFIED", "path": unified_bgc_path, "climatology": True},
    type="bgc",
    use_dask=True,
)

In [ ]:
filepath = project+model_name+"_surface_forcing2024.nc"

In [ ]:
%time surface_forcing.save(filepath, group=True)

In [ ]:
filepath = project+model_name+"_bgc_surface_forcing.nc"

In [ ]:
%time unified_bgc_surface_forcing.save(filepath)

Next we generate the initial file

In [ ]:
from roms_tools import RiverForcing, Grid

In [ ]:
from datetime import datetime

In [ ]:
start_time = datetime(2024, 1, 1)
end_time = datetime(2024, 12, 31)

In [ ]:
river_forcing = RiverForcing(
    grid=grid,
    start_time=start_time,
    end_time=end_time,
    model_reference_date=datetime(2000, 1, 1), # this is the default
    include_bgc=True,
    source = {
    "name": "DAI",
    "path": "/anvil/projects/x-ees250129/Datasets/Iceland_river_dataset/Hvalfjordur_rivers_2024.nc",
    "climatology": False
}
    
)

In [ ]:
river_forcing.ds

In [ ]:
river_forcing.plot_locations()

In [ ]:
river_forcing.plot("river_volume")

In [ ]:
filepath = project+model_name+"_rivers.nc"

In [ ]:
river_forcing.save(filepath=filepath)

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
riv=xr.open_dataset(filepath)
riv.load()

In [ ]:
# Create new array
new_tracer = np.zeros_like(riv['river_tracer'].values)

# Temperature varies sinusoidally with time
temp = 10 * np.sin(np.linspace(0, np.pi, riv.dims['river_time']))  # 12-month cycle

# Loop through tracers
for i, tracer in enumerate(riv['tracer_name'].values):
    if tracer == 'temp':
        new_tracer[:, i, :] = temp[:, np.newaxis]  # vary with time
    elif tracer == 'salt':
        new_tracer[:, i, :] = 1
    elif tracer == 'PO4':
        new_tracer[:, i, :] = 0.4
    elif tracer == 'NO3':
        new_tracer[:, i, :] = 6
    elif tracer == 'SiO3':
        new_tracer[:, i, :] = 3
    elif tracer == 'NH4':
        new_tracer[:, i, :] = 0.4
    elif tracer == 'Fe':
        new_tracer[:, i, :] = 0.000197
    elif tracer == 'Lig':
        new_tracer[:, i, :] = 0.000465    
    elif tracer == 'O2':
        new_tracer[:, i, :] = 360    
    elif tracer == 'DIC':
        new_tracer[:, i, :] = 313    
    elif tracer == 'DIC_ALT_CO2':
        new_tracer[:, i, :] = 313  
    elif tracer == 'ALK':
        new_tracer[:, i, :] = 282 
    elif tracer == 'ALK_ALT_CO2':
        new_tracer[:, i, :] = 282 
    else:
        new_tracer[:, i, :] = 0.0
riv['river_tracer'] = (riv['river_tracer'].dims, new_tracer)

In [ ]:
riv.to_netcdf(project+model_name+"_rivers_modified.nc")

## Halfing the river flux

In [ ]:
import xarray as xr

# Open your modified file
river = xr.open_dataset(project+model_name+"_rivers_modified.nc")
river.load()
# Show what tracers exist
river['river_tracer'].isel(ntracers=10).isel(nriver=6).plot()



In [ ]:
riv['river_volume'][:] = riv['river_volume'].isel(river_time=7)*0.5

In [ ]:
riv.to_netcdf(project+model_name+"_rivers_modified_RHALF.nc")

## CDR release

In [ ]:
from roms_tools import VolumeRelease
from datetime import datetime

In [ ]:
times = [
    datetime(2024, 2, 17, 12, 0),
    datetime(2024, 2, 18, 12, 0),
    datetime(2024, 2, 19, 12, 0),
    datetime(2024, 2, 20, 12, 0),
    datetime(2024, 2, 21, 12, 0),
    datetime(2024, 2, 22, 12, 0),
    datetime(2024, 2, 23, 12, 0),
    datetime(2024, 2, 24, 12, 0),
    datetime(2024, 2, 25, 12, 0),
    datetime(2024, 2, 26, 12, 0),
    datetime(2024, 2, 27, 12, 0),
]

In [ ]:
constant_volume_release_iceland = VolumeRelease(
    name="iceland_release",
    lat=64.394213,  # degree N
    lon=-21.465904,  # degree E
    depth=2,  # m
    times=times,
    volume_fluxes=[0,0.0005,0.0005,0.0005,0.0005,0,0,0,0,0,0],  # m3/s
    tracer_concentrations={
        "temp": 10.0,  # degrees C±±
        "salt": 1.0,  # psu
        "ALK": 1180000  # meq/m3
        },
    fill_values="zero"
)

In [ ]:
constant_volume_release_iceland

In [ ]:
from roms_tools import CDRForcing

In [ ]:
start_time_release = datetime(2024, 2, 17)
end_time_release = datetime(2024, 2, 28)

In [ ]:
cdr_forcing_with_volume_releases1 = CDRForcing(
    grid=grid,
    start_time=start_time_release,
    end_time=end_time_release,
    model_reference_date=datetime(2000, 1, 1), # this is the default
    releases=[
        constant_volume_release_iceland],
)

In [ ]:
cdr_forcing_with_volume_releases.plot_locations()  # By default, this plots all available releases (but max 20).


In [ ]:
cdr_forcing_with_volume_releases1.plot_volume_flux()

In [ ]:
filepath3 = project+model_name+'_cdr_forcing.nc'

In [ ]:
cdr_forcing_with_volume_releases.save(filepath=filepath)

In [ ]:
times = [
    datetime(2024, 2, 17, 12, 0),
    datetime(2024, 2, 18, 12, 0),
    datetime(2024, 2, 19, 12, 0),
    datetime(2024, 2, 20, 12, 0),
    datetime(2024, 2, 21, 12, 0),
    datetime(2024, 2, 22, 12, 0),
    datetime(2024, 2, 23, 12, 0),
    datetime(2024, 2, 24, 12, 0),
    datetime(2024, 2, 25, 12, 0),
    datetime(2024, 2, 26, 12, 0),
    datetime(2024, 2, 27, 12, 0),
]

In [ ]:
constant_volume_release_iceland = VolumeRelease(
    name="iceland_release",
    lat=64.394213,  # degree N
    lon=-21.465904,  # degree E
    depth=2,  # m
    times=times,
    volume_fluxes=[0,0.0005,0.0005,0.0005,0.0005,0,0,0,0,0,0],  # m3/s
    tracer_concentrations={
        "temp": 10.0,  # degrees C±±
        "salt": 1.0,  # psu
        "ALK": 1180000  # meq/m3
        },
    fill_values="zero"
)

In [ ]:
partition_netcdf(filepath, 16, 16,'/home/x-uheede/S/Iceland3_MARBL_2024_60m_CDR/P_INPUT')

In [ ]:
times = [
    datetime(2024, 2, 20, 12, 0),
    datetime(2024, 2, 21, 12, 0),
    datetime(2024, 2, 22, 12, 0),
    datetime(2024, 2, 23, 12, 0),
    datetime(2024, 2, 24, 12, 0),
    datetime(2024, 2, 25, 12, 0),
    datetime(2024, 2, 26, 12, 0),
    datetime(2024, 2, 27, 12, 0),
    datetime(2024, 2, 28, 12, 0),
    datetime(2024, 2, 29, 12, 0),
    datetime(2024, 3, 01, 12, 0),
]

In [ ]:
constant_volume_release_iceland = VolumeRelease(
    name="iceland_release",
    lat=64.394213,  # degree N
    lon=-21.465904,  # degree E
    depth=2,  # m
    times=times,
    volume_fluxes=[0,0.000583,0.000583,0.000583,0.000583,0,0,0,0,0,0],  # m3/s
    tracer_concentrations={
        "temp": 10.0,  # degrees C
        "salt": 1.0,  # psu
        "ALK": 1800000  # meq/m3
        },
    fill_values="zero"
)

In [ ]:
constant_volume_release_iceland

In [ ]:
from roms_tools import CDRForcing

In [ ]:
start_time_release = datetime(2024, 2, 17)
end_time_release = datetime(2024, 2, 28)

In [ ]:
cdr_forcing_with_volume_releases = CDRForcing(
    grid=grid,
    start_time=start_time_release,
    end_time=end_time_release,
    model_reference_date=datetime(2000, 1, 1), # this is the default
    releases=[
        constant_volume_release_iceland],
)

In [ ]:
cdr_forcing_with_volume_releases.plot_locations()  # By default, this plots all available releases (but max 20).


In [ ]:
cdr_forcing_with_volume_releases.plot_volume_flux()

In [ ]:
filepath = project+model_name+'_cdr_forcing_ens1.nc'

In [ ]:
cdr_forcing_with_volume_releases.save(filepath=filepath)

In [ ]:
partition_netcdf(filepath, 16, 16,'/home/x-uheede/S/Iceland3_MARBL_2024_60m_CDR/P_INPUT/')

In [ ]:
times = [
    datetime(2024, 2, 20, 12, 0),
    datetime(2024, 2, 21, 12, 0),
    datetime(2024, 2, 22, 12, 0),
    datetime(2024, 2, 23, 12, 0),
    datetime(2024, 2, 24, 12, 0),
    datetime(2024, 2, 25, 12, 0),
    datetime(2024, 2, 26, 12, 0),
    datetime(2024, 2, 27, 12, 0),
    datetime(2024, 2, 28, 12, 0),
    datetime(2024, 2, 29, 12, 0),
    datetime(2024, 3, 01, 12, 0),
]

In [ ]:
constant_volume_release_iceland = VolumeRelease(
    name="iceland_release",
    lat=64.394213,  # degree N
    lon=-21.465904,  # degree E
    depth=2,  # m
    times=times,
    volume_fluxes=[0,0.000583,0.000583,0.000583,0.000583,0,0,0,0,0,0],  # m3/s
    tracer_concentrations={
        "temp": 10.0,  # degrees C
        "salt": 1.0,  # psu
        "ALK": 1800000  # meq/m3
        },
    fill_values="zero"
)

In [ ]:
from roms_tools import CDRForcing